# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadfarhan2157-source/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Lane 2 is ranking/scoring, so I want a model that outputs a probability/score, not a hard class — evaluated with Precision@50 to match how the queue is actually used (a reviewer works down a ranked list). Per the lane guide's method menu, I'm training Logistic Regression (interpretable baseline-of-models) and Random Forest (the starter pipeline already showed random forest beating hand rules by ~3x on Precision 50), then comparing both against my Week-4 hand-written rule score on the same split and metric — not against each other in isolation.

In [ ]:
candidate_methods = ["Logistic Regression", "Decision Tree", "Random Forest", "Gradient Boosting"]
chosen = ["Logistic Regression", "Random Forest"]
print("Chosen:", chosen, "— evaluated on Precision@50, compared against Week-4 baseline rule")

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Client-grouped, time-aware split. Features come from March 2026, the label comes from April 2026 (strictly future, no overlap with features) — matching the leak lesson from w03. Split is grouped by client_hash_id so no client's pages appear in both train and test; without this, the model could memorize client-level baseline behavior instead of learning transferable signal, which is exactly the risk the lane guide flags for this data.

In [ ]:
path_march = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
path_april = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

from sklearn.model_selection import GroupShuffleSplit

march = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           AVG(content_age_days) AS content_age_days,
           AVG(impressions) AS impressions_march,
           AVG(clicks) AS clicks_march,
           AVG(avg_position) AS avg_position_march,
           SUM(clicks)/NULLIF(SUM(impressions),0) AS ctr_march
    FROM '{path_march}' GROUP BY 1,2
""").df()

april = con.sql(f"""
    SELECT content_hash_id, client_hash_id, AVG(clicks) AS clicks_april
    FROM '{path_april}' GROUP BY 1,2
""").df()

data = march.merge(april, on=["content_hash_id", "client_hash_id"])
data["declined"] = (data["clicks_april"] < data["clicks_march"]).astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=1)
train_idx, test_idx = next(gss.split(data, groups=data["client_hash_id"]))
train, test = data.iloc[train_idx], data.iloc[test_idx]

overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print("Train rows:", len(train), "Test rows:", len(test), "Client overlap (should be 0):", len(overlap))

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

feature_cols = ["content_age_days", "impressions_march", "avg_position_march", "ctr_march"]

X_train, y_train = train[feature_cols], train["declined"]
X_test, y_test = test[feature_cols], test["declined"]

# Week-4 baseline rule, applied as a score on this same test set
test = test.copy()
test["stale_flag"] = ((test["content_age_days"] >= 180) & (test["impressions_march"] >= 500)).astype(int)
test["ctr_gap_flag"] = ((test["avg_position_march"] > 0) & (test["avg_position_march"] <= 20) &
                         (test["ctr_march"] < 0.02) & (test["impressions_march"] >= 500)).astype(int)
baseline_score = 0.5 * test["stale_flag"] + 0.5 * test["ctr_gap_flag"]

def precision_at_k(y_true, scores, k=50):
    top_k_idx = pd.Series(scores).nlargest(k).index
    return y_true.iloc[top_k_idx].mean()

logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, random_state=1).fit(X_train, y_train)

logreg_scores = logreg.predict_proba(X_test)[:, 1]
rf_scores = rf.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "method": ["Week-4 baseline rule", "Logistic Regression", "Random Forest"],
    "roc_auc": [
        roc_auc_score(y_test, baseline_score),
        roc_auc_score(y_test, logreg_scores),
        roc_auc_score(y_test, rf_scores),
    ],
    "precision_at_50": [
        precision_at_k(y_test.reset_index(drop=True), baseline_score.reset_index(drop=True)),
        precision_at_k(y_test.reset_index(drop=True), pd.Series(logreg_scores)),
        precision_at_k(y_test.reset_index(drop=True), pd.Series(rf_scores)),
    ],
})
results


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances)

test_eval = test.copy()
test_eval["rf_score"] = rf_scores
test_eval["actual"] = y_test.values
false_positives = test_eval[(test_eval["rf_score"] > 0.7) & (test_eval["actual"] == 0)]
false_negatives = test_eval[(test_eval["rf_score"] < 0.3) & (test_eval["actual"] == 1)]
print("False positives (high score, didn't decline):", len(false_positives))
print("False negatives (low score, did decline):", len(false_negatives))


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.